# UC2 Extension: τ Sensitivity Analysis

**Question**: Is the design rule — *deploy BO only when SNR > τ* — qualitatively robust to the choice of threshold τ?

**Motivation**: τ = 2.0 is a theoretically-motivated operating point derived from GP posterior variance theory (see `uc2_dag_reasoning.md §4`). It is not an empirically-validated universal constant. This notebook tests whether the qualitative conclusion holds across τ ∈ {1.0, 1.5, 2.0, 2.5, 3.0}.

**What 'qualitatively robust' means here**:
1. The BO advantage is **statistically indistinguishable from zero** below τ (p > 0.05) for all τ values tested.
2. The BO advantage is **significantly positive** above τ (p ≪ 0.001) for all τ values tested.
3. The naive ATE exceeds the true ATE for all τ (confounding direction is stable).

**Important caveat**: The correct claim is *'BO advantage collapses to near zero below τ'*, not *'BO is worse than RS below τ'*. The sigmoid floor (−0.05) is asymptotic and only reached at SNR → 0; the simulated minimum SNR (~0.14) gives f ≈ −0.032, and the mean f(snr) below τ is near zero for all τ in the grid.

**Model specification (within-stratum OLS)**: All stratified ATE estimates in this notebook use the formula:
```
latent_outcome ~ strategy + log_snr_std + budget + dimensionality + initial_design_quality
```
`log_snr_std` is included as a covariate within each stratum to control for residual within-stratum SNR confounding (BO campaigns within a stratum still have higher SNR than RS campaigns on average). Without this covariate, the above-threshold ATE is approximately 2× the true f(snr) due to residual confounding. The values reported here are the **post-fix** estimates; earlier diagnostic cells in the execution trace (`execution_trace.ipynb`) used a simpler model without `log_snr_std` and produced higher above-threshold ATEs (~+0.18–0.20) — those are superseded by the values in this notebook.

> **Note**: All data are synthetic, generated by `src/simulator.py` encoding the UC2 causal DAG. This is a simulation study by design.


In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

from src.simulator import run_campaigns, snr_modulation_fn
from src.graphs import UC2_SNR_THRESHOLD, UC2_BO_FLOOR, UC2_BO_CEILING

sns.set_theme(style="ticks", font_scale=1.1)

TAU_GRID    = [1.0, 1.5, 2.0, 2.5, 3.0]
SEED, N     = 42, 2000
RESULTS_DIR = "/mnt/results"
TAU_COLORS  = ["#4393c3", "#74add1", "#f46d43", "#d73027", "#a50026"]

print(f"Canonical threshold: τ = {UC2_SNR_THRESHOLD}")
print(f"Sensitivity grid:    τ ∈ {TAU_GRID}")
print(f"N campaigns per run: {N}  (same seed={SEED} for all τ)")
print()
print("Midpoint of f(snr; τ) = (BO_FLOOR + BO_CEILING) / 2 =",
      (UC2_BO_FLOOR + UC2_BO_CEILING) / 2)
print("This midpoint is the same for all τ — only the SNR at which it occurs shifts.")


Canonical threshold: τ = 2.0
Sensitivity grid:    τ ∈ [1.0, 1.5, 2.0, 2.5, 3.0]
N campaigns per run: 2000  (same seed=42 for all τ)

Midpoint of f(snr; τ) = (BO_FLOOR + BO_CEILING) / 2 = 0.025
This midpoint is the same for all τ — only the SNR at which it occurs shifts.


## 1. Moderation curves f(snr; τ)

The sigmoid shape is identical for all τ — only the x-position of the midpoint shifts.
The floor (−0.05) and ceiling (+0.10) are fixed by the UC2 DGP regardless of τ.


In [2]:
fig, ax = plt.subplots(figsize=(8, 5))
snr_grid = np.logspace(-1, 2.5, 500)

for tau, color in zip(TAU_GRID, TAU_COLORS):
    lw  = 2.8 if tau == 2.0 else 1.6
    ls  = "-"  if tau == 2.0 else "--"
    lab = f"τ = {tau:.1f}" + (" (canonical)" if tau == 2.0 else "")
    f   = snr_modulation_fn(snr_grid, tau=tau)
    ax.plot(snr_grid, f, color=color, lw=lw, ls=ls, label=lab)
    ax.scatter([tau], [snr_modulation_fn(np.array([tau]), tau=tau)[0]],
               color=color, s=55, zorder=5)

ax.axhline(0, color="grey", lw=1, ls=":")
ax.axhline(UC2_BO_CEILING, color="black", lw=0.8, ls=":", alpha=0.4)
ax.axhline(UC2_BO_FLOOR,   color="black", lw=0.8, ls=":", alpha=0.4)
ax.text(0.12, UC2_BO_CEILING + 0.003, "ceiling (+0.10)", fontsize=8, color="grey")
ax.text(0.12, UC2_BO_FLOOR  - 0.006, "floor (−0.05)",   fontsize=8, color="grey")
ax.set_xscale("log")
ax.set_xlabel("Signal-to-Noise Ratio (SNR)", fontsize=12)
ax.set_ylabel("f(snr; τ)  =  BO advantage", fontsize=12)
ax.set_title("Moderation function shifts with τ\n(dots mark the midpoint at snr = τ)", fontsize=12)
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0.1, 300)
ax.set_ylim(-0.07, 0.13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/uc2_tau_sensitivity_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Sensitivity grid: stratified ATEs across τ values

For each τ, we split campaigns into below-τ and above-τ strata and estimate the
adjusted ATE (OLS on `latent_outcome`, controlling for `{budget, dimensionality,
initial_design_quality}`). We compare to the true ATE = E[f(snr; τ) | stratum].

**Note on below-threshold stratum size**: At τ=1.0, only ~3% of campaigns (n≈53)
fall below threshold. CIs are wide at small τ. This is expected — the design rule
is about identifying the lower tail, not a 50/50 split.


In [3]:
records = []
for tau in TAU_GRID:
    df = run_campaigns(n_campaigns=N, seed=SEED, use_case="UC2", tau=tau)
    df["log_snr"]     = np.log(df["snr"])
    df["log_snr_std"] = (df["log_snr"] - df["log_snr"].mean()) / df["log_snr"].std()

    below = df[df["snr"] <  tau]
    above = df[df["snr"] >= tau]

    def stratum_stats(sub):
        if len(sub) < 20 or sub["strategy"].nunique() < 2:
            return dict(b=np.nan, lo=np.nan, hi=np.nan, p=np.nan, true_f=np.nan)
        # Include log_snr_std to control for within-stratum SNR variation.
        # Without it, the within-stratum OLS is still confounded by SNR
        # (BO campaigns within each stratum have higher SNR than RS campaigns),
        # causing the adjusted ATE to be ~2x the true f(snr).
        # With log_snr_std, the estimate is much closer to the true f(snr).
        formula = ("latent_outcome ~ strategy + log_snr_std "
                   "+ budget + dimensionality + initial_design_quality")
        m = smf.ols(formula, data=sub).fit()
        ci = m.conf_int().loc["strategy"]
        return dict(b=m.params["strategy"], lo=ci[0], hi=ci[1],
                    p=m.pvalues["strategy"], true_f=sub["snr_modulation"].mean())

    sb = stratum_stats(below)
    sa = stratum_stats(above)
    naive = (df.loc[df.strategy==1,"latent_outcome"].mean()
           - df.loc[df.strategy==0,"latent_outcome"].mean())

    records.append({
        "tau":          tau,
        "true_ate":     df["snr_modulation"].mean(),
        "naive_ate":    naive,
        "frac_below":   (df["snr"] < tau).mean(),
        "n_below":      len(below),
        "ate_below":    sb["b"],  "ci_below_lo": sb["lo"], "ci_below_hi": sb["hi"],
        "p_below":      sb["p"],  "true_f_below": sb["true_f"],
        "ate_above":    sa["b"],  "ci_above_lo": sa["lo"], "ci_above_hi": sa["hi"],
        "p_above":      sa["p"],  "true_f_above": sa["true_f"],
    })

res = pd.DataFrame(records)

# Print summary table
print("=== Stratified ATE summary (OLS controls: log_snr_std + budget + dim + idq) ===\n")
print(f"{'τ':>5}  {'n<τ':>6}  {'true f<τ':>10}  {'adj ATE<τ':>11}  "
      f"{'p<τ':>7}  {'true f≥τ':>10}  {'adj ATE≥τ':>11}  {'p≥τ':>12}")
print("-" * 90)
for _, row in res.iterrows():
    p_below_str = f"{row['p_below']:.3f}" if not np.isnan(row['p_below']) else "  n/a"
    p_above_str = f"{row['p_above']:.2e}" if not np.isnan(row['p_above']) else "  n/a"
    print(f"{row['tau']:>5.1f}  {int(row['n_below']):>6}  "
          f"{row['true_f_below']:>+10.4f}  {row['ate_below']:>+11.4f}  "
          f"{p_below_str:>7}  {row['true_f_above']:>+10.4f}  "
          f"{row['ate_above']:>+11.4f}  {p_above_str:>12}")

print()
print("Robustness checks:")
print(f"  Below-threshold ATE not significant (p>0.05): "
      f"{(res['p_below'] > 0.05).all()}  (all τ)")
print(f"  Above-threshold ATE significant (p<0.001):    "
      f"{(res['p_above'] < 0.001).all()}  (all τ)")
print(f"  Naive ATE > true ATE (confounding stable):    "
      f"{(res['naive_ate'] > res['true_ate']).all()}  (all τ)")
print(f"  ATE gap (above − below) > 0:                  "
      f"{((res['ate_above'] - res['ate_below']) > 0).all()}  (all τ)")
print()
print("Note: adj ATE above threshold may still exceed true f(snr) above threshold")
print("because BO campaigns within the above-stratum have higher SNR than RS campaigns.")
print("log_snr_std controls for this; the true f(snr) column is the ground truth.")


=== Stratified ATE summary (OLS controls: log_snr_std + budget + dim + idq) ===

    τ     n<τ    true f<τ    adj ATE<τ      p<τ    true f≥τ    adj ATE≥τ           p≥τ
------------------------------------------------------------------------------------------
  1.0      53     +0.0114      -0.0056    0.932     +0.0925      +0.1141      7.41e-46
  1.5     111     +0.0061      -0.0065    0.833     +0.0916      +0.1139      1.47e-43
  2.0     194     +0.0013      +0.0030    0.867     +0.0916      +0.1144      1.20e-40
  2.5     277     -0.0047      +0.0080    0.564     +0.0918      +0.1138      3.47e-37
  3.0     349     -0.0112      -0.0001    0.991     +0.0917      +0.1139      1.62e-34

Robustness checks:
  Below-threshold ATE not significant (p>0.05): True  (all τ)
  Above-threshold ATE significant (p<0.001):    True  (all τ)
  Naive ATE > true ATE (confounding stable):    True  (all τ)
  ATE gap (above − below) > 0:                  True  (all τ)

Note: adj ATE above threshold may sti

## 3. Figures: stratified ATE panel and true ATE vs τ


In [4]:
# Figure 2: Stratified ATE panel
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
tau_vals = res["tau"].values

# Panel A: Below-threshold
ax = axes[0]
ax.axhline(0, color="crimson", lw=1.5, ls="--", label="Zero (no BO advantage)", zorder=2)
ax.plot(tau_vals, res["true_f_below"], color="#2ca02c", lw=2, ls="-.",
        marker="^", ms=7, label="True E[f(snr) | snr < τ]", zorder=4)
ax.errorbar(tau_vals, res["ate_below"],
            yerr=[res["ate_below"] - res["ci_below_lo"],
                  res["ci_below_hi"] - res["ate_below"]],
            fmt="o-", color="#4393c3", lw=2, ms=7, capsize=5, capthick=1.5,
            label="Adj. ATE (OLS, 95% CI)", zorder=5)
for _, row in res.iterrows():
    if not np.isnan(row["p_below"]):
        sig = "n.s." if row["p_below"] > 0.05 else f"p={row['p_below']:.2f}"
        ax.annotate(sig, xy=(row["tau"], row["ci_below_hi"] + 0.01),
                    ha="center", fontsize=8, color="#4393c3")
ax.fill_between(tau_vals, res["ci_below_lo"], res["ci_below_hi"],
                alpha=0.12, color="#4393c3")
ax.set_xlabel("Threshold τ", fontsize=12)
ax.set_ylabel("ATE (latent outcome units)", fontsize=12)
ax.set_title("Below-threshold: BO advantage\ncollapses to near zero", fontsize=11)
ax.legend(fontsize=9, loc="upper right")
ax.set_xticks(tau_vals)
ax.set_ylim(-0.25, 0.35)
for _, row in res.iterrows():
    ax.annotate(f"n={int(row['n_below'])}", xy=(row["tau"], -0.22),
                ha="center", fontsize=8, color="grey")
ax.text(np.mean(tau_vals), -0.245, "campaigns below τ →", ha="center",
        fontsize=8, color="grey", style="italic")

# Panel B: Above-threshold
ax = axes[1]
ax.axhline(0, color="crimson", lw=1.5, ls="--", zorder=2)
ax.plot(tau_vals, res["true_f_above"], color="#2ca02c", lw=2, ls="-.",
        marker="^", ms=7, label="True E[f(snr) | snr ≥ τ]", zorder=4)
ax.errorbar(tau_vals, res["ate_above"],
            yerr=[res["ate_above"] - res["ci_above_lo"],
                  res["ci_above_hi"] - res["ate_above"]],
            fmt="o-", color="#d73027", lw=2, ms=7, capsize=5, capthick=1.5,
            label="Adj. ATE (OLS, 95% CI)", zorder=5)
for _, row in res.iterrows():
    ax.annotate("p<0.001", xy=(row["tau"], row["ci_above_hi"] + 0.003),
                ha="center", fontsize=8, color="#d73027")
ax.fill_between(tau_vals, res["ci_above_lo"], res["ci_above_hi"],
                alpha=0.12, color="#d73027")
ax.set_xlabel("Threshold τ", fontsize=12)
ax.set_ylabel("ATE (latent outcome units)", fontsize=12)
ax.set_title("Above-threshold: BO advantage\nrobustly positive across all τ", fontsize=11)
ax.legend(fontsize=9, loc="upper right")
ax.set_xticks(tau_vals)
ax.set_ylim(-0.05, 0.30)

plt.suptitle("τ Sensitivity: qualitative design rule holds for τ ∈ {1.0, 1.5, 2.0, 2.5, 3.0}",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/uc2_tau_sensitivity_stratified.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 3: True ATE and fraction-below vs τ
fig, ax1 = plt.subplots(figsize=(8, 5))
color_ate, color_frac = "#4393c3", "#d73027"
ax1.plot(res["tau"], res["true_ate"], "o-", color=color_ate, lw=2.5, ms=8,
         label="True ATE = E[f(snr; τ)]")
ax1.fill_between(res["tau"], res["true_ate"] - 0.005, res["true_ate"] + 0.005,
                 alpha=0.15, color=color_ate)
ax1.set_xlabel("Threshold τ", fontsize=12)
ax1.set_ylabel("True ATE = E[f(snr; τ)]", color=color_ate, fontsize=12)
ax1.tick_params(axis="y", labelcolor=color_ate)
ax1.set_xticks(res["tau"])
ax1.set_ylim(0.06, 0.11)
ax2 = ax1.twinx()
ax2.plot(res["tau"], res["frac_below"] * 100, "s--", color=color_frac,
         lw=2, ms=8, label="% campaigns below τ")
ax2.set_ylabel("Campaigns below τ (%)", color=color_frac, fontsize=12)
ax2.tick_params(axis="y", labelcolor=color_frac)
ax2.set_ylim(0, 30)
canonical_row = res[res["tau"] == 2.0].iloc[0]
ax1.annotate("canonical\nτ = 2.0",
             xy=(2.0, canonical_row["true_ate"]),
             xytext=(2.3, canonical_row["true_ate"] + 0.008),
             arrowprops=dict(arrowstyle="->", color="grey"),
             fontsize=9, color="grey")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc="upper right")
ax1.set_title("True ATE decreases as τ increases\n"
              "(more campaigns fall below threshold, pulling E[f(snr)] down)", fontsize=11)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/uc2_tau_sensitivity_ate_vs_tau.png", dpi=150, bbox_inches="tight")
plt.show()


In [5]:
# Dynamic summary table — computed from res DataFrame above
# (avoids stale hardcoded numbers if parameters change)
print("=== τ Sensitivity: Robustness Summary ===\n")
print(f"{'τ':>5}  {'n<τ':>6}  {'true f<τ':>10}  {'adj ATE<τ':>11}  "
      f"{'p<τ':>12}  {'true f≥τ':>10}  {'adj ATE≥τ':>11}  {'p≥τ':>12}")
print("-" * 95)
for _, row in res.iterrows():
    p_b = f"{row['p_below']:.3f} (n.s.)" if row['p_below'] > 0.05 else f"{row['p_below']:.3f}"
    p_a = f"{row['p_above']:.2e}"
    print(f"{row['tau']:>5.1f}  {int(row['n_below']):>6}  "
          f"{row['true_f_below']:>+10.4f}  {row['ate_below']:>+11.4f}  "
          f"{p_b:>12}  {row['true_f_above']:>+10.4f}  "
          f"{row['ate_above']:>+11.4f}  {p_a:>12}")

print()
print("Robustness verdict:")
print(f"  Below-threshold ATE not significant (p>0.05): {(res['p_below'] > 0.05).all()} for all τ")
print(f"  Above-threshold ATE significant (p<0.001):    {(res['p_above'] < 0.001).all()} for all τ")
print(f"  Naive ATE > true ATE (confounding stable):    {(res['naive_ate'] > res['true_ate']).all()} for all τ")
print()
print("Dissertation framing:")
print("  The design rule — deploy BO only when SNR > τ — is qualitatively robust")
print("  to the choice of τ across {1.0, 1.5, 2.0, 2.5, 3.0}. For all values tested,")
print("  the BO advantage is statistically indistinguishable from zero below the")
print("  threshold and significantly positive above it.")
print()
print("Precision of claim:")
print("  The correct statement is 'BO advantage collapses to near zero below τ',")
print("  NOT 'BO is worse than RS below τ'. The sigmoid floor (−0.05) is asymptotic")
print("  and not reached in the simulated SNR range.")


=== τ Sensitivity: Robustness Summary ===

    τ     n<τ    true f<τ    adj ATE<τ           p<τ    true f≥τ    adj ATE≥τ           p≥τ
-----------------------------------------------------------------------------------------------
  1.0      53     +0.0114      -0.0056  0.932 (n.s.)     +0.0925      +0.1141      7.41e-46
  1.5     111     +0.0061      -0.0065  0.833 (n.s.)     +0.0916      +0.1139      1.47e-43
  2.0     194     +0.0013      +0.0030  0.867 (n.s.)     +0.0916      +0.1144      1.20e-40
  2.5     277     -0.0047      +0.0080  0.564 (n.s.)     +0.0918      +0.1138      3.47e-37
  3.0     349     -0.0112      -0.0001  0.991 (n.s.)     +0.0917      +0.1139      1.62e-34

Robustness verdict:
  Below-threshold ATE not significant (p>0.05): True for all τ
  Above-threshold ATE significant (p<0.001):    True for all τ
  Naive ATE > true ATE (confounding stable):    True for all τ

Dissertation framing:
  The design rule — deploy BO only when SNR > τ — is qualitatively robust
  